# 📓 Semana 11 · Dia 4 — Mosaic AI Vector Search

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition (1 endpoint) |
| **Tempo estimado** | 2h |
| **Certificação alvo** | GenAI Engineer Associate |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Índice vetorial + endpoint funcionando |

---


## 📖 Teoria — Mosaic AI Vector Search

O **Mosaic AI Vector Search** (ex-Databricks Vector Search) guarda embeddings e faz busca por similaridade em escala, integrado ao UC.

Componentes:
- **Index**: `DELTA_SYNC` (sincroniza automaticamente com a tabela Delta) ou `MANAGED` (API)
- **Endpoint**: compute da busca (1 na Free Edition)
- **Filtros de metadata**: refinar por colunas (ex.: categoria)

O índice fica em `workspace.catalog.schema` — governado como qualquer tabela.


### 💻 Na prática — Preparando a tabela com embeddings

Crie a tabela Delta com a coluna de embedding.


In [ ]:
# Tabela com embeddings (chunk por produto)
from pyspark.sql.functions import concat, lit, col
df = (spark.table("workspace.prata.dim_produto")
    .filter(col("Description").isNotNull())
    .limit(500)
    .withColumn("texto", concat(lit("Produto: "), col("Description"),
                                lit(" | Código: "), col("StockCode")))
    .select("StockCode", "texto"))
df.write.mode("overwrite").saveAsTable("workspace.prata.produtos_rag")
print("Tabela produtos_rag criada:", df.count(), "documentos")

In [ ]:
# Gerar embeddings e salvar (use a FMA disponível)
from mlflow.deployments import get_deploy_client
from pyspark.sql.functions import udf, array
from pyspark.sql.types import ArrayType, DoubleType
client = get_deploy_client("databricks")
def embed(textos):
    r = client.predict(endpoint="databricks-bge-large-en", inputs={"input": textos})
    return [d["embedding"] for d in r["data"]]
# Em produção, use o SDK de embeddings em batch; aqui demonstramos a chamada
print("Função de embedding pronta (usada no job de indexação).")

### 💻 Na prática — Criando o índice

Pela UI: **AI → Vector Search → Create Index** (DELTA_SYNC, fonte `workspace.prata.produtos_rag`, coluna de embedding `embedding`).


In [ ]:
# Criar índice via SQL (onde suportado)
sql_index = """
CREATE OR REPLACE VECTOR INDEX workspace.prata.produtos_rag_index
ON TABLE workspace.prata.produtos_rag
INDEX COLUMNS (texto)
SYNC (AUTO)
"""
print(sql_index)
print("Na Free, crie pela UI: AI > Vector Search > Create Index.")

### 💻 Na prática — Buscando

Consulte o índice com a query de exemplo.


In [ ]:
# Busca vetorial via SDK
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient()
resultado = vsc.similarity_search(
    index_name="workspace.prata.produtos_rag_index",
    query_text="copo de vidro",
    columns=["StockCode", "texto"],
    num_results=3)
for r in resultado.get("result", {}).get("data_array", []):
    print(r)

> 🎯 **Dica de prova**: GenAI Assoc: Vector Search com DELTA_SYNC (sincroniza com a tabela Delta) vs MANAGED (API) é pergunta clássica. Na Free: 1 endpoint.


## 🎯 Exercícios de fixação

**1.** Diferença entre índice DELTA_SYNC e MANAGED?

**2.** Como adicionar filtro por categoria na busca?

**3.** O que acontece quando a tabela Delta muda com DELTA_SYNC?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Tipos

DELTA_SYNC: espelha automaticamente a tabela Delta (recomendado). MANAGED: você gerencia os vetores via API.

**2.** Filtro

Passe `filters_json` com a coluna de metadata na similarity_search.

**3.** Delta muda

O índice sincroniza automaticamente (SYNC AUTO) — novos/chunks alterados aparecem sem reprocessar tudo.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*